# V19 fix1 qnli: rerun ONLY the baselines that hit HF 429 (Kwon seeds 77,123,2024; K-prune seeds 42,77,123), then d2i-RI seeds 42,77,123,2024 (seed 7 exists). Tag v19-freeze-fix1; method unchanged

In [ ]:
%%bash
set -e
cd /kaggle/working && rm -rf quantum
git clone -q https://github.com/gourav-ai-engineer/quantum.git
cd quantum && git checkout -q v19-freeze-fix1 && git log --oneline -1 | tee /kaggle/working/commit.txt
export MODEL_NAME=distilbert-base-uncased BL_SKIP_CKPT=0 CK_REQUIRE=qnli BL_TASKS=qnli BL_CONS=0.5 PREP_ARGS=--no-ckpt
export CKPT_DIR_ROOT=/tmp/bl/ck_kprune/distilbert-base-uncased CKPT_SHA_FILE=$PWD/scripts/V19_CKPT_SHA256.json
export BL_KWON_SEEDS="77 123 2024" BL_KPRUNE_SEEDS="42 77 123"
bash scripts/v15_kaggle.sh bl /tmp/bl /kaggle/working/fix_qnli || echo BL_FAILED
python -c "import transformers,torch;assert transformers.__version__=='4.51.3',transformers.__version__;print(torch.__version__,transformers.__version__)" | tee /kaggle/working/env.txt
python scripts/v19_merge_masks.py /kaggle/working/fix_qnli/bl/kprune_masks_merged.npz results/v19/fix1_inputs/qnli_orig_kprune_masks.npz /kaggle/working/fix_qnli/bl/kprune_masks.npz || echo MERGE_FAILED
export BEAT_ARGS="--methods d2i_ri --tasks qnli --constraints 0.5 --seeds 42,77,123,2024 --kprune-masks /kaggle/working/fix_qnli/bl/kprune_masks_merged.npz --ckpt-root $CKPT_DIR_ROOT"
bash scripts/v15_kaggle.sh beat /tmp/bl /kaggle/working/fix_qnli || echo BEAT_FAILED
cd /kaggle/working && tar czf fix_qnli_results.tgz fix_qnli commit.txt env.txt 2>/dev/null || true
ls -la /kaggle/working | head -20
